> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 11 — RL foundations and a trading environment

**Sessions:** S21–S22 (RL foundations · Trading environment design) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Solve a small Markov decision process with value iteration.
2. See tabular Q-learning overfit one price path and generalize from many.
3. Write the reward of a trading environment with the right timing and costs.
4. Compute the differential Sharpe ratio, a reward that cares about risk.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. Value iteration

With known transitions `P[a, s, s']` and rewards `R[s, a]`, the values solve `V(s) = max_a [R(s, a) + γ Σ P(s'|s, a) V(s')]`. Iterate: `Q = R + γ · einsum("ast,t->sa", P, V)`, `V = max over actions`, until `V` stops changing. A two-state market: investing earns 1 in the calm state and loses 2 in the volatile one; states persist.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def value_iteration(P, R, gamma=0.9, tol=1e-10):
    V = np.zeros(P.shape[1])
    while True:
        Q = R + gamma * np.einsum("ast,t->sa", P, V)
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, Q.argmax(axis=1)
        V = V_new

# states: 0 = calm, 1 = volatile; actions: 0 = invest, 1 = cash (the action doesn't change the market)
P = np.array([[[0.9, 0.1], [0.2, 0.8]], [[0.9, 0.1], [0.2, 0.8]]])
R = np.array([[1.0, 0.0], [-2.0, 0.0]])                  # investing earns 1 in calm, loses 2 in volatile
cases = [(P, R, 0.9), (P, R, 0.0), (np.array([np.eye(2), [[0, 1], [1, 0]]]), np.array([[0.0, 0.0], [0.0, 1.0]]), 0.9)]
mine = [p.attempt(value_iteration, *c) for c in cases]
mine = p.check("value_iteration", mine, [p.value_iteration(*c) for c in cases])
V, policy = mine[0]
print(f"values: calm {V[0]:.2f}, volatile {V[1]:.2f};  policy: calm → {['invest', 'cash'][policy[0]]}, volatile → {['invest', 'cash'][policy[1]]}")

Invest when calm, sit in cash when volatile: obvious here, but the same equation solves problems whose answer isn't obvious. In trading we rarely know `P` and `R`, so we **learn** from experience instead.

## 2. Q-learning: one path or many?

Tabular Q-learning on a mean-reverting OU path: the state is the price level in 17 buckets, the actions are short, flat or long, the reward is the next price change (`p.q_learning_paths`). Train once on a single 100-step path for 200 epochs, once on 200 different paths for one epoch, and test both on 300 new paths:

In [ ]:
EDGES = np.linspace(-4, 4, 17)

def ou_path(n, seed, theta=0.2):
    rng = np.random.default_rng(seed)
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = x[t - 1] * (1 - theta) + rng.normal()
    return x

one, many = [ou_path(100, 0)], [ou_path(100, s) for s in range(10, 210)]
new = [ou_path(100, s) for s in range(1000, 1300)]
q_one = p.q_learning_paths(one, EDGES, epochs=200)
q_many = p.q_learning_paths(many, EDGES, epochs=1)
pd.DataFrame({"one path, 200 epochs": {"in sample": p.greedy_reward(q_one, one, EDGES), "new paths": p.greedy_reward(q_one, new, EDGES)},
              "200 paths, 1 epoch": {"in sample": p.greedy_reward(q_many, many, EDGES), "new paths": p.greedy_reward(q_many, new, EDGES)}}).round(3)

Trained on one history, the agent memorizes it: a great in-sample reward that falls apart on new paths. Trained on many paths it earns less in sample and keeps it out of sample. A backtest is one path; RL needs many (simulated or bootstrapped) paths.

## 3. The environment

A Gymnasium-style environment (`reset()`, `step(action)`) over daily returns `r`. The observation at step `t` contains returns **up to** `t−1`; the action sets the new position (−1, 0, +1), which then earns `r[t]`, the return **after** the decision. Trading costs `cost × |position change|`. Write the reward.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyTradingEnv(p.TradingEnv):
    def step(self, action):
        new_pos = float(action) - 1.0                                # action 0/1/2 → position −1/0/+1
        reward = new_pos * self.r[self.t] - self.cost * abs(new_pos - self.pos)
        self.pos, self.t = new_pos, self.t + 1
        terminated = self.t >= self.end
        obs = self._obs() if not terminated else np.zeros(self.L + 1)
        return obs, float(reward), terminated, False, {}

returns = p.ar1(4000, 0.3, 0.01, seed=0)                 # AR(1) returns with φ = 0.3: momentum at lag 1
train, test = returns[:3000], returns[3000:]
actions = np.random.default_rng(0).integers(0, 3, 300)

def rewards(env):
    env.reset()
    return [env.step(a)[1] for a in actions]

mine = p.check("TradingEnv.step", p.attempt(rewards, MyTradingEnv(test, cost=0.0005)), rewards(p.TradingEnv(test, cost=0.0005)))
print(f"300 random actions: total reward {np.sum(mine):.4f}")

Before any agent, run simple rules through the environment on the test period (observations scaled with the **training** volatility):

In [ ]:
rules = {"flat": lambda o: 1, "long only": lambda o: 2, "momentum": lambda o: 2 if o[-2] > 0 else 0,
         "reversal": lambda o: 0 if o[-2] > 0 else 2}
env = p.TradingEnv(test, cost=0.0002, scale=train.std())
pd.Series({k: p.run_policy(env, f) for k, f in rules.items()}, name="total test reward (2 bp costs)").round(3).to_frame()

The data has momentum at lag 1, and the one-line momentum rule harvests it. Any agent will be judged against this number (notebook 12).

## 4. A reward that cares about risk

Summing P&L rewards risk-taking. Moody & Saffell's **differential Sharpe ratio** is the marginal contribution of today's return `R_t` to an exponentially weighted Sharpe ratio. Track `A` (mean) and `B` (second moment), both starting at 0: `ΔA = R_t − A`, `ΔB = R_t² − B`, then `D_t = (B·ΔA − ½·A·ΔB) / (B − A²)^{3/2}` using the **previous** `A` and `B` (`D_t = 0` while `B − A² <= 0`), and finally `A += η·ΔA`, `B += η·ΔB`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def differential_sharpe(returns, eta=0.01):
    A = B = 0.0
    out = []
    for R in np.asarray(returns, dtype=float):
        dA, dB = R - A, R * R - B
        den = B - A * A
        out.append((B * dA - 0.5 * A * dB) / den ** 1.5 if den > 0 else 0.0)
        A, B = A + eta * dA, B + eta * dB
    return np.array(out)

steady = np.r_[np.full(60, 0.001) + np.random.default_rng(1).normal(0, 0.002, 60), -0.03, np.full(20, 0.001)]
cases = [(steady,), (test[:500], 0.05)]
mine = [p.attempt(differential_sharpe, *c) for c in cases]
mine = p.check("differential_sharpe", mine, [p.differential_sharpe(*c) for c in cases])
print(f"day 60 (a −3% shock after steady gains): D = {mine[0][60]:.1f};  a typical steady day: D = {np.median(mine[0][30:60]):.2f}")

A loss after a calm run is punished far harder than a gain is rewarded: the reward sees the risk. Use it (or a P&L reward with a variance penalty) when the agent's position size matters.

## Wrap-up

* Value iteration when the model is known; learning when it isn't.
* One history is one sample: train on many paths.
* The reward at `t` uses the return **after** the decision, net of costs; scale observations with training statistics.
* Graded version: `labs/part10/week38_rl` (`value_iteration`, `q_learning_paths`, `TradingEnv` with Gymnasium, `differential_sharpe`).